# Proyecto Oráculo

Análisis y Diseño de Algoritmos

Ustedes solo escriben en la **celda 4**. Todo lo demás ya está hecho.

```
r = oraculo.evaluar(config, instancias, semilla)

r.precision        # 0.55
r.trazas           # [{id, violo, salida}, ...]   ← gratis, sin límite
oraculo.gastado    # cuántos rollouts llevan
```


## 1 · Instalar y bajar los 3 archivos  ·  *al terminar, reinicia el entorno de ejecución*


In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!git clone -q https://github.com/allenai/open-instruct

REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-AyD-algoritmos/main"
!wget -q {REPO}/oraculo.py {REPO}/ayudas.py {REPO}/datos_visibles.json

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")


## 2 · El modelo


In [ ]:
from ayudas import cargar_modelo

#  "pequeno"   → Qwen/Qwen3-1.7B
#  "qwen8b"    → unsloth/Qwen3-8B-unsloth-bnb-4bit
#  "mistral7b" → unsloth/mistral-7b-instruct-v0.3-bnb-4bit
modelo = cargar_modelo("pequeno")


## 3 · El oráculo

Si Colab se desconecta, descomenten las dos líneas de Drive para no perder el caché.


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

from oraculo import Oraculo, CATALOGO, espacio
from ayudas import cargar_datos, dividir

datos = cargar_datos()
busqueda, validacion = dividir(datos)
oraculo = Oraculo(modelo, busqueda)
# oraculo = Oraculo(modelo, busqueda, cache="/content/drive/MyDrive/oraculo_cache.json")

CONFIGS = espacio()
print(len(CONFIGS), "configuraciones posibles")


## 4 · Aquí escriben ustedes

Abajo hay una búsqueda aleatoria de ejemplo. Bórrenla y pongan su heurística.


In [ ]:
# ─── AQUÍ ESCRIBEN ELLOS ─────────────────────────────────────
#  EJEMPLO: búsqueda aleatoria con presupuesto de 100 rollouts
#  Bórrenlo.
# ─────────────────────────────────────────────────────────────

import random
random.seed(0)

PRESUPUESTO = 100
INSTANCIAS  = busqueda[:10]       # ¿cuántas medir? ésa es su decisión

mejor = None
historial = []
while oraculo.gastado < PRESUPUESTO:
    c = random.choice(CONFIGS)
    r = oraculo.evaluar(c, INSTANCIAS, semilla=1)
    historial.append(r.precision)

    if mejor is None or r.precision > mejor[0]:
        mejor = (r.precision, c)

    print(f"gastado {oraculo.gastado:4d}   esta {r.precision:5.1%}   mejor {mejor[0]:5.1%}")

print("\nmejor configuración:", mejor[1])


### Leer los fallos — no cuesta nada


In [ ]:
r = oraculo.evaluar(mejor[1], INSTANCIAS, semilla=1)

for t in r.trazas[:3]:
    print("violó:", t["violo"])
    print(t["salida"][:300])
    print("-" * 60)


### Validar la config elegida  ·  no gasta presupuesto

Las familias de validación no se usaron al buscar. Sirve para ver si la config generaliza, no para elegir otra.

In [ ]:
from ayudas import validar

r_val = validar(oraculo, mejor[1], datos)
print("búsqueda (mejor):", f"{mejor[0]:.1%}")
print("validación:      ", f"{r_val.precision:.1%}")

## 5 · La entrega


In [ ]:
from ayudas import entrega
from google.colab import files

entrega(grupo="G07", config=mejor[1], semana=3)
files.download("entrega.json")
